In [1]:
##Here, the process of cleaning up excess text in the dataset, whether caused by API or other issues, has been performed.

In [ ]:
import os
import json

base_path = r'C:\Users\Faruk\Desktop\PROJELER\IDE\AI_Generated_text\Multi_API_System' # File path

total_files_scanned = 0
total_records_found = 0
total_duplicate_records = 0
total_cleaned_records = 0

print("Scanning folders, cleaning and saving files individually...\n" + "-"*50)

for root, dirs, files in os.walk(base_path):
    for file in files:
        # Read only json/jsonl files and skip previously created "Temizlenmis_" files
        if (file.lower().endswith('.json') or file.lower().endswith('.jsonl')) and not file.startswith('Temizlenmis_'):
            file_path = os.path.join(root, file)
            total_files_scanned += 1
            print(f"Processing: {os.path.basename(root)} / {file}")
            
            seen_family_ids_in_file = set()
            file_cleaned_data = []
            file_duplicates = 0
            
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    for line_number, line in enumerate(f, 1):
                        line = line.strip()
                        if not line:
                            continue
                            
                        try:
                            record = json.loads(line)
                            total_records_found += 1
                            
                            family_id = record.get("family_id")
                            
                            if family_id:
                                if family_id not in seen_family_ids_in_file:
                                    seen_family_ids_in_file.add(family_id)
                                    file_cleaned_data.append(record)
                                else:
                                    file_duplicates += 1
                                    total_duplicate_records += 1
                            else:
                                pass # Skip records without family_id
                                
                        except json.JSONDecodeError:
                            print(f"  -> Error: {file} line {line_number} is not a valid JSON.")
                            
                # Reading and filtering of the file is completed, now save it to its own folder
                if file_cleaned_data:
                    # Determine the new file name (Example: Temizlenmis_gemini_verisi.json)
                    clean_file_name = f"Temizlenmis_{file}"
                    clean_file_path = os.path.join(root, clean_file_name)
                    
                    with open(clean_file_path, 'w', encoding='utf-8') as out_f:
                        json.dump(file_cleaned_data, out_f, ensure_ascii=False, indent=4)
                        
                    total_cleaned_records += len(file_cleaned_data)
                    print(f"  -> Success: {len(file_cleaned_data)} records kept, {file_duplicates} duplicates removed.")
                    print(f"  -> Saved: {clean_file_name}\n")
                    
            except Exception as e:
                print(f"File read/write error ({file_path}): {e}")

print("-" * 50)
print("ALL OPERATIONS COMPLETED! General Summary:")
print(f"Total Files Scanned: {total_files_scanned}")
print(f"Total Records Read: {total_records_found}")
print(f"Total Duplicates Removed: {total_duplicate_records}")
print(f"Total Clean Records Saved: {total_cleaned_records}")
print("-" * 50)

In [ ]:
## HERE, THE CLEANED JSON FILES ARE COLLECTED UNDER A SINGLE FOLDER, NOT IN DIFFERENT FOLDERS:

In [ ]:
import os
import shutil

base_path = r'C:\Users\Faruk\Desktop\PROJELER\IDE\AI_Generated_text\Multi_API_System'
target_folder = os.path.join(base_path, 'Cleaned_Data_Pool')

# Create the pool folder if it does not exist
if not os.path.exists(target_folder):
    os.makedirs(target_folder)

print(f"'{os.path.basename(target_folder)}' folder is ready. Collecting files...\n" + "-"*50)

moved_file_count = 0

for root, dirs, files in os.walk(base_path):
    # Prevent reading files inside the newly created pool folder again
    if target_folder in root:
        continue
        
    for file in files:
        # Target only the cleaned files
        if file.startswith('Temizlenmis_'):
            source_path = os.path.join(root, file)
            
            # Get the folder name of the model it came from
            model_folder = os.path.basename(root)
            
            # Some file names may be the same (e.g., Generated.jsonl).
            # Add the folder name (model name) to the file name to prevent overwriting.
            new_file_name = f"{model_folder}_{file}"
            target_path = os.path.join(target_folder, new_file_name)
            
            # Copy the file to the new pool together with its original metadata
            shutil.copy2(source_path, target_path)
            moved_file_count += 1
            
            print(f"Collected -> {new_file_name}")

print("-" * 50)
print(f"A total of {moved_file_count} cleaned files have been collected separately in the pool.")

In [ ]:
##Here, we separate the cleaned dataset into 200 text files based on score ranges. The texts are not selected randomly, with selected same Family Id

In [ ]:
import os
import json
import pandas as pd

ROOT_FOLDER = r'C:\Users\Faruk\Desktop\PROJELER\IDE\AI_Generated_text\Multi_API_System\Temizlenmis_Veri_Havuzu'
CENTRAL_OUTPUT_FOLDER = r'C:\Users\Faruk\Desktop\PROJELER\IDE\AI_Generated_text\Multi_API_System\Final_Sample_200'
os.makedirs(CENTRAL_OUTPUT_FOLDER, exist_ok=True)

TARGET_TOTAL = 200
TARGET_PER_GROUP = 50
REFERENCE_SAMPLE_FILE = os.path.join(
    CENTRAL_OUTPUT_FOLDER,
    "reference_sample_ids.json"
)

def extract_original_scores(item):
    if 'json_data' in item:
        root = item['json_data']
        for v in root.get('versions', []):
            if v.get('type') == 'original' and 'original_scores' in v:
                return v['original_scores']
        if 'source' in root and 'original_scores' in root['source']:
            return root['source']['original_scores']
    if 'source' in item and 'original_scores' in item['source']:
        return item['source']['original_scores']
    return None

def create_reference_sample(data):
    rows = []
    for item in data:
        scores = extract_original_scores(item)
        if not scores:
            continue
        try:
            quality_score = (
                scores['grammar'] +
                scores['vocabulary'] +
                scores['cohesion'] +
                scores['syntax']
            ) / 4
        except KeyError:
            continue
        if quality_score < 2.5:
            group = '1_Low (1.0-2.5)'
        elif quality_score < 3.0:
            group = '2_MidLow (2.5-3.0)'
        elif quality_score < 3.5:
            group = '3_MidHigh (3.0-3.5)'
        else:
            group = '4_High (3.5-5.0)'
        rows.append({
            "family_id": item.get("family_id"),
            "score": quality_score,
            "group": group
        })

    df = pd.DataFrame(rows)
    sampled_ids = []
    
    for group_name in sorted(df['group'].unique()):
        subset = df[df['group'] == group_name]
        if len(subset) < TARGET_PER_GROUP:
            sampled = subset
        else:
            sampled = subset.sample(
                n=TARGET_PER_GROUP,
                random_state=42
            )
        sampled_ids.extend(
            sampled['family_id'].tolist()
        )
    with open(
        REFERENCE_SAMPLE_FILE,
        'w',
        encoding='utf-8'
    ) as f:
        json.dump(
            sampled_ids,
            f,
            ensure_ascii=False,
            indent=4
        )
    print(
        f"Reference sample created: {len(sampled_ids)} texts"
    )
    return set(sampled_ids)

def process_single_file(file_path, reference_ids):
    base_filename = os.path.splitext(
        os.path.basename(file_path)
    )[0]
    output_sampled_path = os.path.join(
        CENTRAL_OUTPUT_FOLDER,
        f"{base_filename}_subset_200.jsonl"
    )

    if os.path.exists(output_sampled_path):
        print(
            f"Skipping existing file: {base_filename}"
        )
        return

    try:
        with open(
            file_path,
            'r',
            encoding='utf-8'
        ) as f:
            data = json.load(f)
            
    except Exception as e:
        print(
            f"File reading error ({base_filename}): {e}"
        )
        return

    selected_data = []
    for item in data:
        family_id = item.get("family_id")
        if family_id in reference_ids:
            selected_data.append(item)

    with open(
        output_sampled_path,
        'w',
        encoding='utf-8'
    ) as f:
        for item in selected_data:
            f.write(
                json.dumps(
                    item,
                    ensure_ascii=False
                )
                + '\n'
            )
    print(
        f"SUCCESS: {base_filename} -> {len(selected_data)} identical samples selected."
    )

def main():
    print("=" * 70)
    print(
        "AI TEXT SAMPLING SYSTEM STARTED"
    )
    print(
        f"Source Folder: {ROOT_FOLDER}"
    )
    print(
        f"Output Folder: {CENTRAL_OUTPUT_FOLDER}"
    )
    print("=" * 70)
    files = [
        f for f in os.listdir(ROOT_FOLDER)
        if f.endswith('.json')
        or f.endswith('.jsonl')
    ]
    if not files:
        print(
            "ERROR: No files found!"
        )
        return

    # Create common sample only once
    if os.path.exists(
        REFERENCE_SAMPLE_FILE
    ):
        with open(
            REFERENCE_SAMPLE_FILE,
            'r',
            encoding='utf-8'
        ) as f:
            reference_ids = set(
                json.load(f)
            )
        print(
            "Existing reference sample loaded."
        )

    else:
        first_file = os.path.join(
            ROOT_FOLDER,
            files[0]
        )
        with open(
            first_file,
            'r',
            encoding='utf-8'
        ) as f:
            first_data = json.load(f)
        reference_ids = create_reference_sample(
            first_data
        )
    processed_count = 0

    for file in files:
        file_path = os.path.join(
            ROOT_FOLDER,
            file
        )
        print(
            f"\nProcessing: {file}"
        )
        process_single_file(
            file_path,
            reference_ids
        )
        processed_count += 1

    print("\n" + "=" * 70)
    print(
        f"COMPLETED! {processed_count} model files processed using the same 200 texts."
    )
    print("=" * 70)

if __name__ == "__main__":

    main()